# Finetune Gemma 4 E2B for German spell checking

This is the **Gemma 4** variant of `9_finetune_llm.ipynb`. Same task, same data, same recipe - but with Google's `gemma-4-E2B-it` instead of `Llama-3.2-1B-Instruct`. In our tests Gemma 4 E2B corrects noticeably more sentences than the 1B models, at the price of a bigger download (10 GB) and slower training on a Tesla T4.

The notebook is written for a **free Tesla T4 Google Colab instance**: press "*Runtime*" and then "*Run all*".

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), how to save it and how to run it with [Ollama](#Ollama).

This notebook is based on Unsloth's [Gemma 4 (E2B) Text](https://github.com/unslothai/notebooks) and [Llama3 (8B) Ollama](https://github.com/unslothai/notebooks) templates (LGPL-3.0). To install Unsloth on your own machine, follow [their guide](https://unsloth.ai/docs/get-started/install).

### Installation

Gemma 4 needs a newer `transformers` than the Llama notebook, so the version pins below differ.

Running this on your own machine? Use Python 3.12 or 3.13 - the pinned `datasets` version does not work on Python 3.14 yet.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth matplotlib  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio
import torch; torch._dynamo.config.recompile_limit = 64;

### Load the model

`gemma-4-E2B-it` is Google's smallest Gemma 4 instruct model. "E2B" means *effective 2B*: the model has 5.1 billion parameters on disk, but a large part of them are per-layer embeddings that are cheap to compute, so it runs roughly like a 2B model. It understands text, images and audio - we only use the text part.

**T4 note:** Gemma models do not work in float16, so Unsloth trains them in float32 on a T4 (which has no bfloat16). Expect the training to be slower than in the Llama notebook. The 10 GB download takes a minute or two on Colab.

In [ ]:
from unsloth import FastModel
import torch
max_seq_length = 1024 # Our examples are short: 99% of the input + output pairs have fewer than 300 tokens.
load_in_4bit = False  # 4 bit quantization saves memory - set to True if you run out of memory on a T4.

gemma4_models = [
    "unsloth/gemma-4-E2B-it",  # our choice: effective 2B, 5.1B parameters on disk
    "unsloth/gemma-4-E4B-it",  # bigger sibling, effective 4B
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

We now add LoRA adapters so we only need to update a small amount of parameters! Gemma 4 is a multimodal model, so we tell Unsloth to leave the vision layers alone and only finetune the language part.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep

Our dataset is a CSV file with 10,000 German sentences. The column `text` contains a sentence with (synthetic) spelling mistakes, the column `summary` the correct version. We load it directly from the workshop's GitHub repository, so this notebook also runs in Colab without uploading anything.

We keep 200 examples aside as a test set - later we use them to check what the model has learned.

In [ ]:
from datasets import load_dataset

csv_url = "https://raw.githubusercontent.com/oliverguhr/workshop-ki-deepdive/main/data/fix.spelling.mini.csv"
# csv_url = "./data/fix.spelling.mini.csv" # use this if you run the notebook locally inside the workshop repository

dataset = load_dataset("csv", data_files = csv_url, split = "train")
dataset = dataset.rename_columns({"text": "input", "summary": "output"})
dataset = dataset.train_test_split(test_size = 200, seed = 3407)
train_dataset, test_dataset = dataset["train"], dataset["test"]
print(train_dataset)
train_dataset[0]

For `Ollama` and `llama.cpp` to work like a chatbot, a training example must consist of exactly two parts: what the user says and what the model answers.

Unsloth's `to_sharegpt` builds these pairs. `merged_prompt` is the user message - here it is simply the misspelled sentence (the instruction "please correct this" goes into the system prompt in the next step, so that later on you can just paste a sentence into Ollama). `output_column_name` is the answer the model should learn.

Spell checking is a single turn task, so we set `conversation_extension = 1`. Unsloth's template uses `3` to glue random examples together into fake multi-turn chats - we do not want that here.

In [ ]:
from unsloth import to_sharegpt, standardize_sharegpt

train_dataset = to_sharegpt(
    train_dataset,
    merged_prompt = "{input}",
    output_column_name = "output",
    conversation_extension = 1, # 1 = single turn. No fake multi-turn chats for spell checking.
)
train_dataset = standardize_sharegpt(train_dataset)
train_dataset[0]["conversations"]

### Chat template

Now we define how a training example looks as plain text. We use the **Gemma 4 chat format** with its `<|turn>` and `<turn|>` markers, because that is what `gemma-4-E2B-it` was trained with. Unsloth fills in `{SYSTEM}`, `{INPUT}` and `{OUTPUT}`.

The system prompt tells the model what its job is. Later we also put it into the Ollama `Modelfile`, so in Ollama you can simply paste a sentence and get the correction back.

(Unsloth also ships this format as a built-in: `get_chat_template(tokenizer, chat_template = "gemma-4")`. We spell it out here so you can see what the model actually reads.)

In [ ]:
from unsloth import apply_chat_template

system_prompt = "Du bist ein Korrekturprogramm. Korrigiere alle Rechtschreib- und Grammatikfehler im Text des Nutzers und antworte nur mit dem korrigierten Text."

chat_template = """<bos><|turn>system
{SYSTEM}<turn|>
<|turn>user
{INPUT}<turn|>
<|turn>model
{OUTPUT}<turn|>
"""

train_dataset = apply_chat_template(
    train_dataset,
    tokenizer = tokenizer,
    chat_template = chat_template,
    default_system_message = system_prompt,
)
print(train_dataset[0]["text"])

### Before training

Let's see how the model does *before* finetuning. The freshly added LoRA adapters are initialised with zeros, so right now the model still behaves exactly like the original `gemma-4-E2B-it`.

Unlike the 1B models, Gemma 4 E2B is already a usable spell checker out of the box. Look closely though: the base model likes to drop or change words (a `so` here, a hyphen there) and invents names. After finetuning it sticks much closer to the input - even if it will not be perfect after 150 steps either.

`korrigiere()` sends a sentence through the chat template and lets the model generate a correction. We use greedy decoding (`do_sample = False`): for spell checking we want the most likely correction, not a creative one.

We look at four test sentences with typical typos (swapped, missing and doubled letters, wrong capitalisation). The dataset also contains a harder kind of noise where every `e` became an `ä` - have a look at `test_dataset[0]` and `test_dataset[2]` if you want to see how the model copes with that.

In [ ]:
# For multimodal models like Gemma 4, `tokenizer` is a processor that also handles images and audio.
# The plain text tokenizer lives inside it - that is all we need for spell checking.
text_tokenizer = tokenizer.tokenizer

def korrigiere_batch(texts, max_new_tokens = 256):
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": text}],
            tokenize = False,
            add_generation_prompt = True, # Must add for generation
        )
        for text in texts
    ]
    inputs = text_tokenizer(
        prompts,
        return_tensors = "pt",
        padding = True,
        padding_side = "left", # all prompts of a batch must end right where the model starts writing
        add_special_tokens = False,
    ).to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False, # greedy decoding: always take the most likely token
            temperature = 1.0, top_p = 1.0, top_k = None, # keeps transformers from warning about unused sampling settings
        )
    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:] # cut off the prompt
    return [text_tokenizer.decode(ids, skip_special_tokens = True).strip() for ids in new_tokens]

def korrigiere(text):
    return korrigiere_batch([text])[0]

def zeige_beispiele(examples):
    for example in examples:
        print("Input:     ", example["input"])
        print("Correction:", korrigiere(example["input"]))
        print("Expected:  ", example["output"])
        print("-" * 100)

test_examples = test_dataset.select([1, 3, 6, 7])
FastModel.for_inference(model) # Enable native 2x faster inference
zeige_beispiele(test_examples)

### Measuring the error rate during training

The training loss only tells us that the model gets better at predicting the next token. What we actually care about is: **how many spelling errors are left?** So we measure that directly, while the model trains.

`character_error_rate` counts how many characters have to be inserted, deleted or replaced to turn the model's output into the correct sentence (the *Levenshtein distance*), divided by the length of the correct sentence. `0` means perfect. The cell also prints the error rate of the uncorrected input, so we know what we start from.

A `TrainerCallback` runs this on 24 test sentences every 25 steps - and once *before* the first step, so we also see what the untrained model can do. It also reports how many sentences came out exactly right, but with only 24 sentences that percentage jumps by 4 points per sentence, so the character error rate is the number to watch. On a T4 each measurement takes a while (float32 generation); increase `every_n_steps` if you are in a hurry.

In [ ]:
from transformers import TrainerCallback

def edit_distance(a, b):
    """Levenshtein distance: how many single characters must be inserted, deleted or replaced to turn a into b."""
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start = 1):
        current = [i]
        for j, char_b in enumerate(b, start = 1):
            current.append(min(previous[j] + 1,                      # delete char_a
                               current[j - 1] + 1,                   # insert char_b
                               previous[j - 1] + (char_a != char_b)))# replace (free if equal)
        previous = current
    return previous[-1]

def character_error_rate(predictions, references):
    """Edited characters divided by the length of the correct text, averaged over all sentences. 0 = perfect."""
    return sum(edit_distance(p, r) / max(len(r), 1) for p, r in zip(predictions, references)) / len(references)

class CERCallback(TrainerCallback):
    """Corrects the evaluation sentences with the current model every `every_n_steps` steps and records the error rate."""
    def __init__(self, examples, every_n_steps = 25):
        self.examples = examples
        self.every_n_steps = every_n_steps
        self.history = []

    def measure(self, step):
        FastModel.for_inference(model)
        corrections = korrigiere_batch([e["input"] for e in self.examples])
        FastModel.for_training(model)
        cer = character_error_rate(corrections, [e["output"] for e in self.examples])
        exact = sum(c == e["output"] for c, e in zip(corrections, self.examples)) / len(self.examples)
        self.history.append({"step": step, "cer": cer, "exact": exact})
        print(f"step {step:4d}: character error rate {cer:.3f}, {exact:.0%} of the sentences exactly right")

    def on_train_begin(self, args, state, control, **kwargs):
        self.measure(0) # the untrained model

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps == 0:
            self.measure(state.global_step)

    def on_train_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps != 0:
            self.measure(state.global_step) # the final model, if we have not just measured it

eval_examples = test_dataset.select(range(10, 34)) # 24 sentences the model never sees during training
input_cer = character_error_rate([e["input"] for e in eval_examples], [e["output"] for e in eval_examples])
print(f"Error rate of the uncorrected input: {input_cer:.3f}")

<a name="Train"></a>
### Train the model

Now let's train our model. Gemma 4 E2B learns this task fast: in our tests 100 steps already gave a lower character error rate than the Llama 1B model after a full epoch of 1,225 steps. We use `max_steps = 150` with an effective batch size of 8 (4 sentences per step, accumulated over 2 steps to save T4 memory), so the model sees 1,200 examples.

On a T4 this trains in float32 and is slower than the Llama notebook. If it takes too long for you, reduce `max_steps` to 100. For a full run over all 9,800 training examples set `num_train_epochs = 1` and remove `max_steps`.

In [ ]:
from trl import SFTTrainer, SFTConfig
FastModel.for_training(model) # switch back from inference to training mode
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        max_steps = 150,
        # num_train_epochs = 1, # For a full training run - then remove max_steps!
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

We only want to train on the model's *answers*, not on the misspelled input - otherwise the model would also learn to *produce* spelling mistakes. Unsloth's `train_on_responses_only` masks everything up to the model turn, so those tokens do not contribute to the loss. It already knows where the user and model parts start from the chat template we applied above.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer) # Unsloth knows the user/model markers from our chat template

cer_callback = CERCallback(eval_examples, every_n_steps = 25)
trainer.add_callback(cer_callback)

We verify masking is actually done. First the full example - notice there is exactly one `<bos>`:

In [ ]:
text_tokenizer.decode(trainer.train_dataset[0]["input_ids"])

And now only the part the model is trained on - the system prompt and the misspelled input are masked out:

In [ ]:
space = text_tokenizer(" ", add_special_tokens = False).input_ids[0]
text_tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### How the error rate went down

The dashed line is the error rate of the uncorrected input. Everything below it means the model made the text better than it was. Step 0 is the model before finetuning: Gemma 4 already removes about half of the errors out of the box, the finetuning roughly halves what is left.

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in cer_callback.history]
cers  = [h["cer"]  for h in cer_callback.history]

fig, ax = plt.subplots(figsize = (8, 4.5))
ax.plot(steps, cers, color = "#2a78d6", linewidth = 2, marker = "o", markersize = 6, label = "model output")
ax.axhline(input_cer, color = "#52514e", linewidth = 1.5, linestyle = "--", label = "uncorrected input")
for step, cer in [(steps[0], cers[0]), (steps[-1], cers[-1])]:
    ax.annotate(f"{cer:.3f}", (step, cer), textcoords = "offset points", xytext = (6, 6), color = "#52514e")
ax.set_xlabel("training step")
ax.set_ylabel("character error rate")
ax.set_title("Spelling errors left after correction (lower is better)")
ax.set_ylim(bottom = 0)
ax.grid(axis = "y", color = "#e6e5e1")
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.legend(frameon = False)
plt.show()

for h in cer_callback.history:
    print(f"step {h['step']:4d}: character error rate {h['cer']:.3f}, {h['exact']:.0%} exactly right")

<a name="Inference"></a>
### After training

The same four test sentences and the same function as above - but now with the trained LoRA adapters.

In [ ]:
FastModel.for_inference(model) # Enable native 2x faster inference
zeige_beispiele(test_examples)

Try your own sentence:

In [ ]:
print(korrigiere("Das ist ein Satz mit vielen Rechtschreibfelern, den das Modell korigieren sol."))

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to GGUF for Ollama, scroll down!

In [ ]:
model.save_pretrained("gemma-4-e2b-german-spelling-lora")  # Local saving
tokenizer.save_pretrained("gemma-4-e2b-german-spelling-lora")
# model.push_to_hub("your_name/gemma-4-e2b-german-spelling-lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/gemma-4-e2b-german-spelling-lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma-4-e2b-german-spelling-lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        load_in_4bit = load_in_4bit,
    )
    FastModel.for_inference(model) # Enable native 2x faster inference
    print(korrigiere(test_examples[0]["input"]))

<a name="Ollama"></a>
### Ollama Support

[Ollama](https://ollama.com/) can import a model straight from a folder of Safetensors weights ([docs](https://docs.ollama.com/import#importing-a-model-from-safetensors-weights)) and quantize it on the way in. That gives us a spell checker we can run locally on a laptop - without converting anything ourselves.

Let's first install `Ollama`. Two extras are needed on Linux: Ollama runs Gemma 4 through its **MLX runtime**, which the Linux installer does not include (about 1.2 GB), and that runtime needs `libgfortran`.

In [ ]:
# ollama's installer extracts a zstd archive and refuses to run without zstd, which Colab does not ship
!command -v zstd >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y zstd) >/dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
# the MLX runtime (needed for the Safetensors import and to run Gemma 4)
!curl -fsSL https://ollama.com/download/ollama-linux-amd64-mlx.tar.zst | tar --zstd -x -C /usr/local
!apt-get -qq install -y libgfortran5 >/dev/null 2>&1 || true

Next, we merge the LoRA adapters into the base model and save the result as a normal Hugging Face model in 16 bit. Unsloth streams this through the file on disk, so it only needs about 2 GB of RAM. The folder `gemma-4-e2b-german-spelling` (10 GB) is what Ollama will import.

In [ ]:
# Running locally as a normal user and getting "Permission denied: .../model.safetensors"? Unsloth copies the base
# weights from the Hugging Face cache and keeps their read-only flag. Fix:
#   chmod u+w ~/.cache/huggingface/hub/models--unsloth--gemma-4-E2B-it/snapshots/*/model.safetensors
import os
model.save_pretrained_merged("gemma-4-e2b-german-spelling", tokenizer, save_method = "merged_16bit")
print(sorted(os.listdir("gemma-4-e2b-german-spelling")))

**Optional: GGUF export.** Unsloth can also convert the merged model to a `GGUF` file for `llama.cpp`. This does **not** work on a free Colab instance: Gemma 4 keeps its per-layer embeddings in one 4.7 GB tensor, and the converter needs about 15 GB of RAM for it, Colab has 12.7 GB. On a machine with 24 GB RAM or more, set `False` to `True`.

In [ ]:
if False:
    model.save_pretrained_gguf("gemma-4-e2b-german-spelling", tokenizer, quantization_method = "Q8_0")

The finetuned model now lives in the merged folder, so we can free the GPU. On a T4 this matters: Ollama wants to load the 8 GB model into the same GPU memory that our training model still occupies.

In [ ]:
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still reserved: {torch.cuda.memory_reserved() / 1024**3:.1f} GB")

We use `subprocess` to start `Ollama` in a non blocking fashion! On your own computer you can simply open a new terminal and type `ollama serve`, but in Colab we have to use this hack.

In [ ]:
import subprocess
import time

import requests

subprocess.Popen(["ollama", "serve"])

# Wait until Ollama is actually answering instead of guessing how long it needs.
for _ in range(60):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout = 2).ok:
            print("Ollama is ready!")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "Ollama did not become ready on http://localhost:11434 within 60s. "
        "Check the `ollama serve` output above."
    )

`Ollama` needs a `Modelfile`. `FROM` points to the folder with the Safetensors weights, `TEMPLATE` describes the prompt format (in Ollama's Go template syntax - compare it with our chat template above), `SYSTEM` sets the system prompt (that is why you can just paste a sentence into Ollama later) and `PARAMETER temperature 0` makes the spell checker deterministic.

In [ ]:
modelfile = '''FROM ./gemma-4-e2b-german-spelling
TEMPLATE """{{- if .System }}<|turn>system
{{ .System }}<turn|>
{{ end }}
{{- range .Messages }}
{{- if eq .Role "user" }}<|turn>user
{{ .Content }}<turn|>
{{ else if eq .Role "assistant" }}<|turn>model
{{ .Content }}<turn|>
{{ end }}
{{- end }}<|turn>model
"""
SYSTEM """SYSTEM_PROMPT"""
PARAMETER stop "<turn|>"
PARAMETER temperature 0
'''.replace("SYSTEM_PROMPT", system_prompt)

with open("Modelfile", "w") as f:
    f.write(modelfile)
print(modelfile)

Now we create an `Ollama` model called `german-spelling-gemma`. `-q int8` tells Ollama to quantize the weights to 8 bit while importing (the model shrinks from 10 GB to about 8 GB - the embeddings stay in 16 bit). Without `-q` you get the full 16 bit model.

In [ ]:
!ollama create -q int8 german-spelling-gemma -f Modelfile

And now we can do inference on it via the `Ollama` API! The system prompt is part of the Modelfile, so we only send the misspelled sentence.

You can also upload the model to `Ollama` and try the `Ollama` Desktop app, see https://www.ollama.com/

In [ ]:
response = requests.post(
    "http://localhost:11434/api/chat",
    json = {
        "model": "german-spelling-gemma",
        "messages": [{"role": "user", "content": test_examples[0]["input"]}],
        "stream": False,
    },
    timeout = 300,
)
print("Input:     ", test_examples[0]["input"])
print("Correction:", response.json()["message"]["content"])
print("Expected:  ", test_examples[0]["output"])

# Interactive mode

### ⭐ To use the spell checker interactively, first click the **| >_ |** button to open a terminal.
![](https://raw.githubusercontent.com/unslothai/unsloth/nightly/images/Where_Terminal.png)

### ⭐ Then type `ollama run german-spelling-gemma`, paste any German sentence with typos and press `ENTER`. To exit, hit `CTRL + D`.

On your own computer: copy the `gemma-4-e2b-german-spelling` folder and the `Modelfile`, run `ollama create -q int8 german-spelling-gemma -f Modelfile` and then `ollama run german-spelling-gemma`. On macOS Ollama brings the MLX runtime along; on Linux install it as in the installation cell above.

---

This notebook is based on the Unsloth notebooks, which are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme). If you have questions about Unsloth, there is a [Discord](https://discord.gg/unsloth) channel, an [Installation Guide](https://unsloth.ai/docs/get-started/install) and a [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms).